In [9]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [10]:
!pip install -U transformers datasets peft accelerate bitsandbytes evaluate rouge_score wandb

## 1. Подготовка данных

датасет Dolly

In [11]:
from datasets import load_dataset
import json
import re
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

**Функции очистки**

In [12]:
def clean_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def to_markdown_style(response):
    return f"""### Short Answer

{response}

### Key Points

- The answer is structured and easy to scan.
- Important details are separated into clear sections.

### Conclusion

This response follows a clean markdown style."""

**Сборка instruction-response:**

In [13]:
samples = []

seen = set()


for row in dataset:
    instruction = clean_text(row.get("instruction", ""))
    response = clean_text(row.get("response", ""))

    if len(instruction) < 10:
        continue
    if len(response) < 50 or len(response) > 1500:
        continue

    key = instruction.lower()
    if key in seen:
        continue
    seen.add(key)

    styled_response = to_markdown_style(response)

    samples.append({
        "instruction": instruction,
        "response": styled_response
    })

    if len(samples) >= 300:
        break

len(samples)

300

**Сохраняем JSONL**

In [14]:
with open(DATA_DIR / "raw_samples.jsonl", "w", encoding="utf-8") as f:
    for item in samples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

Train/eval split:

In [15]:
train_data, eval_data = train_test_split(samples, test_size=0.1, random_state=42)

with open(DATA_DIR / "train.jsonl", "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(DATA_DIR / "eval.jsonl", "w", encoding="utf-8") as f:
    for item in eval_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(len(train_data), len(eval_data))

270 30


## **2. Prompt format для SFT**

In [16]:
def format_example(example):
    return f"""### Instruction:
{example["instruction"]}

### Response:
{example["response"]}"""

Загружаем dataset

In [17]:
from datasets import load_dataset

data_files = {
    "train": "data/train.jsonl",
    "eval": "data/eval.jsonl"
}

dataset = load_dataset("json", data_files=data_files)

dataset = dataset.map(lambda x: {"text": format_example(x)})

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

## **3. Загрузка модели в 4-bit**

In [18]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## **4. PEFT / LoRA config**

In [19]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## **5. Tokenization**

In [20]:
MAX_LENGTH = 256

def tokenize_function(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(dataset["train"].column_names)

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

## **6. Training**

In [21]:
import os
import wandb

os.environ["WANDB_MODE"] = "online"

wandb.login()
wandb.init(
    project="llm-markdown-style-sft",
    name="qlora-markdown-style-run"
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


...

wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ndarzhanov3 (ndarzhanov3-nwpu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [22]:
import wandb
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

wandb.init(project="llm-markdown-style-sft")

training_args = TrainingArguments(
    output_dir="outputs/markdown-style-adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=80,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_steps=40,
    report_to="wandb",
    run_name="qlora-markdown-style-run",
    save_total_limit=1,
    remove_unused_columns=False
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.123361
10,1.488981
15,1.188477
20,1.414579
25,1.022599
30,1.098393
35,1.128456
40,1.102044
45,1.311405
50,1.172978


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=80, training_loss=1.2735500752925872, metrics={'train_runtime': 147.858, 'train_samples_per_second': 2.164, 'train_steps_per_second': 0.541, 'total_flos': 511467608604672.0, 'train_loss': 1.2735500752925872, 'epoch': 1.1777777777777778})

In [23]:
adapter_dir = "outputs/markdown-style-adapter/final"

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

('outputs/markdown-style-adapter/final/tokenizer_config.json',
 'outputs/markdown-style-adapter/final/chat_template.jinja',
 'outputs/markdown-style-adapter/final/tokenizer.json')

In [24]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU")

CUDA available: True
GPU: Tesla T4


## **7. Inference: base vs fine-tuned**

In [25]:
def generate_answer(model, tokenizer, instruction, max_new_tokens=200):
    prompt = f"""### Instruction:
{instruction}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

In [26]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

base_outputs = []

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [27]:
from peft import PeftModel

ft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

ft_model = PeftModel.from_pretrained(ft_model, adapter_dir)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [28]:
test_instructions = [
    "Explain what overfitting is in machine learning.",
    "Give tips for improving SQL query performance.",
    "Explain the difference between precision and recall.",
    "Write a short guide on how to prepare data for ML training.",
    "Explain what Docker is and why developers use it.",
    "Give a checklist for evaluating a classification model.",
    "Explain what feature engineering means.",
    "Describe the difference between supervised and unsupervised learning.",
    "Give advice on writing clean Python code.",
    "Explain what an API is in simple terms."
]

In [29]:
results = []

for instruction in test_instructions:
    base_answer = generate_answer(base_model, tokenizer, instruction)
    ft_answer = generate_answer(ft_model, tokenizer, instruction)

    results.append({
        "instruction": instruction,
        "base_answer": base_answer,
        "fine_tuned_answer": ft_answer
    })

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/t

## **8. Метрика: ROUGE + кастомный markdown style score**

In [30]:
import evaluate

rouge = evaluate.load("rouge")

references = []
base_predictions = []
ft_predictions = []

for item in results:
    instruction = item["instruction"]

    reference = f"""### Short Answer

A clear answer to: {instruction}

### Key Points

- Structured explanation.
- Easy-to-read markdown format.

### Conclusion

The answer should be concise and well formatted."""

    references.append(reference)
    base_predictions.append(item["base_answer"])
    ft_predictions.append(item["fine_tuned_answer"])

base_rouge = rouge.compute(predictions=base_predictions, references=references)
ft_rouge = rouge.compute(predictions=ft_predictions, references=references)

base_rouge, ft_rouge

({'rouge1': np.float64(0.12060056567014932),
  'rouge2': np.float64(0.025852097154623645),
  'rougeL': np.float64(0.0966397891645831),
  'rougeLsum': np.float64(0.11064376644487141)},
 {'rouge1': np.float64(0.2349843249123792),
  'rouge2': np.float64(0.06352308236369185),
  'rougeL': np.float64(0.16583134649584447),
  'rougeLsum': np.float64(0.22414179112899893)})

In [31]:
def markdown_style_score(text):
    score = 0

    if "###" in text:
        score += 1
    if "-" in text or "*" in text:
        score += 1
    if "Conclusion" in text or "Вывод" in text:
        score += 1
    if len(text.split()) >= 50:
        score += 1

    return score / 4

base_style_scores = [markdown_style_score(x) for x in base_predictions]
ft_style_scores = [markdown_style_score(x) for x in ft_predictions]

base_avg_style = sum(base_style_scores) / len(base_style_scores)
ft_avg_style = sum(ft_style_scores) / len(ft_style_scores)

base_avg_style, ft_avg_style

(0.375, 0.95)

In [32]:
import json
from pathlib import Path

Path("reports").mkdir(exist_ok=True)

eval_report = {
    "base_rouge": base_rouge,
    "fine_tuned_rouge": ft_rouge,
    "base_markdown_style_score": base_avg_style,
    "fine_tuned_markdown_style_score": ft_avg_style,
    "examples": results
}

with open("reports/eval_results.json", "w", encoding="utf-8") as f:
    json.dump(eval_report, f, ensure_ascii=False, indent=2)

In [34]:
with open("reports/examples_comparison.md", "w", encoding="utf-8") as f:
    for i, item in enumerate(results[:10], 1):
        f.write(f"## Example {i}\n\n")
        f.write(f"### Instruction\n\n{item['instruction']}\n\n")
        f.write(f"### Base Model Answer\n\n{item['base_answer']}\n\n")
        f.write(f"### Fine-tuned Model Answer\n\n{item['fine_tuned_answer']}\n\n")
        f.write("---\n\n")